In [2]:
# "INFOF422 Statistical foundations of machine learning" course
## fpe.py
# Author: G. Bontempi
# Python translation of the R package gbcode 

from IPython.display import display, clear_output
import numpy as np
import matplotlib.pyplot as plt
from numpy.linalg import pinv
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
#import ipywidgets as widgets
import matplotlib
import time 

np.random.seed(0)

n = 3  # number of input variables
p = n + 1
p_max = 25
N = 50  # number of training data
x = np.sort(np.random.uniform(-1, 1, N))

X = np.ones((N, 1))
for j in range(1, p_max + 1):
    X = np.hstack((X, x.reshape(-1, 1) ** j))

xts = np.arange(-1, 1.01, 0.01)
Xts = np.ones((len(xts), 1))
for j in range(1, p_max + 1):
    Xts = np.hstack((Xts, xts.reshape(-1, 1) ** j))

beta = np.hstack(([1], np.arange(1, n + 1))).reshape(-1, 1)

sd_w = 0.5

f = X[:, :p] @ beta
Y = f.flatten() + np.random.normal(0, sd_w, N)

fts = Xts[:, :p] @ beta
Yts = fts.flatten() + np.random.normal(0, sd_w, len(fts))

R_emp = []
MISE = []
FPE = []
PSE = []
no_par = []
plt.figure()
for i in range(2, min(p_max, N - 1) + 1):
    
    XX = X[:, :i]
    invX = pinv(XX.T @ XX)
    beta_hat = invX @ XX.T @ Y
    Y_hat = XX @ beta_hat

    XXts = Xts[:, :i]
    Y_hats = XXts @ beta_hat
    no_par.append(i)

    e = Y - Y_hat
    R_emp.append((e.T @ e) / N)
    sde2hat = (e.T @ e) / (N - i)
    
    fig, ax = plt.subplots(figsize=(12, 10))  # Create figure and axes

    ax.plot(xts, fts, label='True function', color='green', linewidth=3)
    ax.scatter(x, Y, label='Data Points', color='blue')
    ax.plot(xts, Y_hats, label='Fitted function', color='red')
    plt.ylim(min(Y), max(Y))
    ax.legend()
    
    e_ts = Yts - Y_hats
    MISE.append((e_ts.T @ e_ts) / N)
    FPE.append(((1 + i / N) / (1 - i / N)) * (e.T @ e) / N)
    PSE.append((e.T @ e) / N + 2 * sde2hat * i / N)
    
    plt.title(f"degree={i-1}; MISE_emp={R_emp[i-2]:.4f}; FPE={FPE[i-2]:.2f}; PSE={PSE[i-2]:.3f}",
              fontsize=18)
    
    display(fig)
    clear_output(wait=True)  # Clear the output
    
    plt.close(fig)  # Close the figure
    time.sleep(0.1)
    input(" ")
    
    
    plt.ioff()  # Disable interactive mode

fig, axs = plt.subplots(2, 2, figsize=(12, 10))
## max p to visualise
p_max2=p_max
axs[0, 0].plot(np.array(no_par) - 1, R_emp, label='Empirical risk')
axs[0, 0].set_xlabel("# parameters")
axs[0, 0].set_ylabel("Empirical risk")
axs[0, 0].set_title("Empirical risk")
axs[0, 0].set_xlim(2, p_max2)
axs[0, 0].set_ylim(0, 0.4)

axs[0, 1].plot(np.array(no_par) - 1, MISE, label='Generalization error')
axs[0, 1].set_xlabel("# parameters")
axs[0, 1].set_ylabel("Generalization error")
axs[0, 1].set_title("Generalization error")
axs[0, 1].set_xlim(2, p_max2)
axs[0, 1].set_ylim(1, 4)

axs[1, 0].plot(np.array(no_par) - 1, FPE, label='FPE')
axs[1, 0].set_xlabel("# parameters")
axs[1, 0].set_ylabel("FPE")
axs[1, 0].set_title("FPE")
axs[1, 0].set_xlim(2, p_max2)
axs[1, 0].set_ylim(0.1, 0.5)

axs[1, 1].plot(np.array(no_par) - 1, PSE, label='PSE')
axs[1, 1].set_xlabel("# parameters")
axs[1, 1].set_ylabel("PSE")
axs[1, 1].set_title("PSE")
axs[1, 1].set_xlim(2, p_max2)
axs[1, 1].set_ylim(0.1, 0.5)

plt.tight_layout()
plt.show()
plt.close()

print(f"which.min(R.emp)={np.argmin(R_emp) + 1}")
print(f"which.min(MISE)={np.argmin(MISE) + 1}")
print(f"which.min(FPE)={np.argmin(FPE) + 1}")
print(f"which.min(PSE)={np.argmin(PSE) + 1}")


KeyboardInterrupt: Interrupted by user

<Figure size 640x480 with 0 Axes>

In [ ]:
# "INFOF422 Statistical foundations of machine learning" course
# biasvar_vis.py
# Author: G. Bontempi

# Visualization of bias variance tradeoff for a set of polynomial fittings

import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import pandas as pd
from IPython.display import clear_output, display
import time

# Define the function f(x, ord) = 1 + x + x^2 + ... + x^ord
def f(x, ord):
    f_val = 1
    for i in range(1, ord + 1):
        f_val += x ** i
    return f_val



# Set random seed
np.random.seed(1)

n = 1
N = 20

x = np.linspace(-2, 2, N)
N = len(x)
sd_w = 0.75
O = 3
Y = f(x, ord=O) + np.random.normal(0, sd_w, size=N)
data_tr = np.column_stack((Y, x))
# %%

# %%


MaxDegree = 15 ## max degree of fitting polynomial

Remp = np.zeros(MaxDegree)  ## Empirical Risk
MSE_loo = np.zeros(MaxDegree) ## Leave-One-Out
x_ts = np.linspace(-2, 2, 200)

B2 = np.zeros(MaxDegree)  ## Squared Bias
V = np.zeros(MaxDegree)  ## Variance
PR = np.zeros(MaxDegree)
FPE = np.zeros(MaxDegree) ## FPE term



for r in range(1, MaxDegree + 1):
    X = np.column_stack([x ** ord for ord in range(1, r + 1)])
    X_ts = np.column_stack([x_ts ** ord for ord in range(1, r + 1)])
    p = r + 1
    Pr = []
    fig, ax = plt.subplots(figsize=(8, 8)) 
    for rr in range(1, 501):
        np.random.seed(rr)
        Y = f(x, ord=O) + np.random.normal(0, sd_w, size=N)
        DN = pd.DataFrame(X, columns=[f'x^{i}' for i in range(1, r + 1)])
        DN['Y'] = Y

        # Add constant term for intercept
        X_with_const = sm.add_constant(X)
        model = sm.OLS(Y, X_with_const).fit()
        sd_w_hat = np.sqrt(np.sum(model.resid ** 2) / (N - p))

        if rr == 1:
            Remp[r - 1] = np.mean(model.resid ** 2)
            PR[r - 1] = Remp[r - 1] 
            FPE[r - 1] = Remp[r - 1] + 2 * sd_w_hat * p / N

        Y_ts = f(x_ts, ord=O)
        data_ts = pd.DataFrame(X_ts, columns=[f'x^{i}' for i in range(1, r + 1)])
        data_ts['Y'] = Y_ts

        if rr == 1:
            ax.plot(x_ts, Y_ts, label='True Function', color='blue', linewidth=2)
            ax.scatter(x, Y, label='Data Points', color='black')
            plt.ylim(min(Y), max(Y))
        pr = model.predict(sm.add_constant(X_ts))
        Pr.append(pr)
        ax.plot(x_ts, pr, color='red', alpha=0.1)

    Pr = np.array(Pr).T
    mean_pr = np.mean(Pr, axis=1)
    ax.plot(x_ts, Y_ts, color='blue', linewidth=2)
    ax.plot(x_ts, mean_pr, color='green', linewidth=2, label='Mean Prediction')

    bias = np.round(np.mean(np.abs(Y_ts - mean_pr)), 2)
    variance = np.round(np.mean(np.var(Pr, axis=1)), 2)
    Remp_val = np.round(Remp[r - 1], 3)

    plt.title(f"N={N}; degree={r}\n Bias={bias}; Var={variance}; Emp risk={Remp_val}")
    B2[r - 1] = np.mean((Y_ts - mean_pr) ** 2)
    V[r - 1] = np.mean(np.var(Pr, axis=1))

    plt.legend()
    display(fig)
    clear_output(wait=True)  # Clear the output
    
    plt.close(fig)  # Close the figure
    time.sleep(0.1) 
    input(" ")
    
    

plt.figure(figsize=(10, 6))
mR = MaxDegree
degrees = np.arange(1, mR + 1)

plt.plot(degrees[:MaxDegree], Remp[:MaxDegree], label='Remp', color='yellow', linewidth=3)
plt.plot(degrees[:MaxDegree], B2[:MaxDegree] + V[:MaxDegree], label='MSE', color='black', linewidth=3)
plt.plot(degrees[:MaxDegree], B2[:MaxDegree], label='Bias', color='green', linewidth=3)
plt.plot(degrees[:MaxDegree], V[:MaxDegree], label='Variance', color='red', linewidth=3)
plt.plot(degrees[:MaxDegree], PR[:MaxDegree], label='LOO', color='orange', linewidth=3)
plt.plot(degrees[:MaxDegree], FPE[:MaxDegree], label='FPE', color='cyan', linewidth=3)

plt.title("Bias-variance tradeoff")
plt.xlabel("Degree")
plt.ylabel("Error Metrics")
plt.legend(loc='upper right')
plt.ylim(-10, 3)
plt.show()


KeyboardInterrupt: Interrupted by user